# Fase 1 · Pipeline ETL, NLP y Enriquecimiento Geoespacial (CABA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Darkops56/CapstoneProject/blob/main/etl/etl_pipeline.ipynb)

Este notebook presenta **paso a paso** el proceso completo de **Extracción, Transformación y Carga (ETL)** del Capstone Project para relocalización corporativa en la Ciudad Autónoma de Buenos Aires (CABA).

### Objetivo de Negocio
Unificar 5 fuentes de datos heterogéneas para construir un dataset limpio, auditable y enriquecido geoespacialmente que permita estimar el **costo mensual real (`costo_total_ars = alquiler_ars + expensas_ars`)** y evaluar la idoneidad de cada propiedad para perfiles **`FAMILIA`** y **`SOLTERO_EJECUTIVO`**.

### Mapa de Pasos
0. **Configuración del Entorno (Local / Google Colab)** y parámetros de negocio.
1. **Extracción y Auditoría Cruda** de publicaciones de Zonaprop (`zonaprop_alquileres_caba_1000.json`).
2. **Unificación Monetaria y Extracción de Expensas** (prevención de *Data Leakage*).
3. **Ingeniería de Atributos NLP**: `apto_mascotas` (con prioridad de negaciones) y `score_amenities` por perfil.
4. **Limpieza Espacial, Coherencia Física e Imputación de Nulos**.
5. **Enriquecimiento Geoespacial con las 4 Capas Urbanas de CABA**:
   - Ruido diurno (`GeoPandas sjoin` + `sjoin_nearest`).
   - Delitos 2022 (`normalize_caba_coord` + `BallTree` Haversine a 500 m).
   - Establecimientos educativos (reproyección Gauss-Krüger → `EPSG:4326` + `BallTree` a 500 m).
   - Bocas de subte (`BallTree` distancia mínima y líneas a 600 m).
6. **Auditoría de Outliers** (IQR `1.5×` y `|Z-score| > 3`).
7. **Exportación de Resultados Curados** y resumen ejecutivo de auditoría.

## 0. Configuración del Entorno (Local y Google Colab)

Esta celda detecta automáticamente si el notebook se ejecuta en **Google Colab** o de forma **local**:
- En Colab, verifica e instala únicamente las librerías faltantes y clona el repositorio (o permite subir los archivos si no están disponibles).
- Define todas las constantes monetarias y geoespaciales configurables (`TIPO_CAMBIO_BASE`, bounding box CABA, proyección local y pesos delictivos).

In [ ]:
import csv
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

IN_COLAB = "google.colab" in sys.modules

# 1. Instalar dependencias solo si faltan (pensado para Google Colab)
REQUIRED_PACKAGES = {
    "geopandas": "geopandas",
    "shapely": "shapely",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
    "seaborn": "seaborn",
    "matplotlib": "matplotlib",
}
missing_pkgs = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    print(f"Instalando dependencias faltantes: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely import wkt
from sklearn.neighbors import BallTree

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

# 2. Localizar raíz del proyecto (Local o Colab)
REPO_URL = "https://github.com/Darkops56/CapstoneProject.git"

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path("/content/CapstoneProject")]
    for cand in candidates:
        if (cand / "Datasets" / "Originales").exists():
            return cand.resolve()
    if IN_COLAB:
        target = Path("/content/CapstoneProject")
        print(f"Clonando repositorio en {target}...")
        res = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], capture_output=True, text=True)
        if res.returncode == 0 and (target / "Datasets" / "Originales").exists():
            return target.resolve()
        print("No se pudo clonar automáticamente (repositorio privado o sin acceso). Puedes montar Drive o subir los archivos.")
    return Path.cwd().resolve()

BASE_DIR = resolve_project_root()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

DATASETS_DIR = BASE_DIR / "Datasets"
DATASETS_ORIGINALES_DIR = DATASETS_DIR / "Originales"
DATASETS_CURADOS_DIR = DATASETS_DIR / "Curados"
PRODUCTO1_DIR = BASE_DIR / "producto1"
PRODUCTO1_DIR.mkdir(parents=True, exist_ok=True)

def resolve_input(filename: str) -> Path:
    for folder in [DATASETS_ORIGINALES_DIR, BASE_DIR, Path.cwd()]:
        p = folder / filename
        if p.exists():
            return p
    return DATASETS_ORIGINALES_DIR / filename

ZONAPROP_PATH = resolve_input("zonaprop_alquileres_caba_1000.json")
RUIDO_PATH = resolve_input("medicion_de_ruido_diurno_2018.csv")
DELITOS_PATH = resolve_input("delitos_2022.xlsx")
ESCUELAS_PATH = resolve_input("establecimientos_educativos.xlsx")
SUBTE_PATH = resolve_input("bocas-de-subte.csv")

# Si faltan archivos en Colab, ofrecer carga manual interactiva
if IN_COLAB and not all(p.exists() for p in [ZONAPROP_PATH, RUIDO_PATH, DELITOS_PATH, ESCUELAS_PATH, SUBTE_PATH]):
    from google.colab import files
    print("Sube los 5 archivos fuente faltantes desde tu equipo:")
    files.upload()
    ZONAPROP_PATH = resolve_input("zonaprop_alquileres_caba_1000.json")
    RUIDO_PATH = resolve_input("medicion_de_ruido_diurno_2018.csv")
    DELITOS_PATH = resolve_input("delitos_2022.xlsx")
    ESCUELAS_PATH = resolve_input("establecimientos_educativos.xlsx")
    SUBTE_PATH = resolve_input("bocas-de-subte.csv")

# 3. Parámetros centralizados de negocio y geoprocesamiento
TIPO_CAMBIO_BASE = 1200.0
FACTOR_IMPUESTOS_USD = 1.0
IMPUESTO_SELLOS_ALQUILER = 0.0
TARGET_COL = "costo_total_ars"

CABA_LAT_MIN, CABA_LAT_MAX = -34.72, -34.50
CABA_LON_MIN, CABA_LON_MAX = -58.58, -58.32
CABA_LOCAL_PROJ = (
    "+proj=tmerc +lat_0=-34.6297166 +lon_0=-58.4627 "
    "+k=0.999998 +x_0=20000 +y_0=70000 +ellps=intl +units=m +no_defs"
)
EARTH_RADIUS_M = 6_371_000.0

CRIME_WEIGHT_HOMICIDIO = 10.0
CRIME_WEIGHT_ARMA = 4.0
CRIME_WEIGHT_ROBO_LESIONES = 2.0
CRIME_WEIGHT_OTROS = 1.0

BARRIO_COMUNA_FALLBACK = {
    "Almagro": 5, "Balvanera": 3, "Belgrano": 13, "Caballito": 6,
    "Colegiales": 13, "Núñez": 13, "Nuñez": 13, "Palermo": 14,
    "Recoleta": 2, "Villa Devoto": 11, "Villa Urquiza": 12,
}

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local'} | BASE_DIR: {BASE_DIR}")
for label, path in [("Zonaprop", ZONAPROP_PATH), ("Ruido", RUIDO_PATH), ("Delitos", DELITOS_PATH), ("Escuelas", ESCUELAS_PATH), ("Subte", SUBTE_PATH)]:
    print(f"  - {label:8s}: {'OK' if path.exists() else 'FALTA'} ({path.name})")

## 1. Extracción y Auditoría Cruda de Zonaprop

Cargamos `zonaprop_alquileres_caba_1000.json` e inspeccionamos la estructura original, el volumen de publicaciones y la distribución inicial de monedas (`ARS` vs `USD`).

In [ ]:
with open(ZONAPROP_PATH, "r", encoding="utf-8") as f:
    raw_items = json.load(f)

print(f"Total de registros crudos en JSON: {len(raw_items)}")

raw_preview = pd.DataFrame([
    {
        "listingId": it.get("listingId"),
        "moneda": it.get("list_price_currency") or "ARS",
        "precio": it.get("list_price_amount"),
        "lat": it.get("list_latitude"),
        "lon": it.get("list_longitude"),
        "ubicacion": (it.get("location") or {}).get("neighborhood") or it.get("list_location_label"),
    }
    for it in raw_items
])

display(raw_preview.head(5))
print("\nDistribución inicial de monedas:")
display(raw_preview["moneda"].value_counts().to_frame("cantidad"))

## 2. Unificación Monetaria (`costo_total_ars`) y Extracción de Expensas

Para evitar **Data Leakage**, definimos como variable objetivo:
$$\text{costo\_total\_ars} = \text{alquiler\_ars} + \text{expensas\_ars}$$

- Los contratos en **USD** se convierten usando `TIPO_CAMBIO_BASE = 1200.0`, `FACTOR_IMPUESTOS_USD = 1.0` e `IMPUESTO_SELLOS_ALQUILER = 0.0`.
- Las **expensas** se extraen en orden de confiabilidad:
  1. Campo estructurado (`prices.detailPricesArray[0].expenses`).
  2. Frases explícitas de `"expensas incluidas"` / `"sin expensas"` (`0.0 ARS`, evitando capturar por error el precio del alquiler previo).
  3. Expresión regular estricta *posterior* a la palabra `"expensas"`.
  4. Las faltantes se imputarán más adelante usando la mediana de `expensas/m²` del barrio.

In [ ]:
def parse_ar_number(raw: str) -> float:
    """Convierte montos con formato argentino ('1.200.000', '382000,50') a float."""
    s = raw.strip().rstrip(".,-")
    if not s:
        return np.nan
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    elif "." in s:
        parts = s.split(".")
        if all(len(p) == 3 for p in parts[1:]):
            s = "".join(parts)
    elif "," in s:
        parts = s.split(",")
        s = s.replace(",", ".") if len(parts[-1]) <= 2 else s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_price_ars(
    amount: Any,
    currency: str | None,
    tipo_cambio: float = TIPO_CAMBIO_BASE,
    factor_usd: float = FACTOR_IMPUESTOS_USD,
    sellos: float = IMPUESTO_SELLOS_ALQUILER,
) -> float:
    """Unifica precios en USD o ARS a Pesos Argentinos (ARS)."""
    if amount is None or (isinstance(amount, float) and np.isnan(amount)):
        return np.nan
    try:
        val = float(amount)
    except (TypeError, ValueError):
        return np.nan
    if val <= 0:
        return np.nan

    curr = (currency or "ARS").strip().upper()
    ars = val * tipo_cambio * factor_usd if curr in {"USD", "U$S", "U$D", "DOLAR", "DOLARES"} else val
    return ars * (1.0 + sellos)


def extract_expenses_ars(
    item: dict[str, Any],
    tipo_cambio: float = TIPO_CAMBIO_BASE,
    factor_usd: float = FACTOR_IMPUESTOS_USD,
) -> tuple[float, str]:
    """Extrae expensas desde campos estructurados o regex posterior a 'expensas'."""
    prices = item.get("prices") or {}
    for dp in (prices.get("detailPricesArray") or []):
        exp_str = dp.get("expenses")
        if exp_str:
            m = re.search(r"(USD|U\$S|\$)?\s*([\d.,]+)", str(exp_str), re.I)
            if m:
                val = parse_ar_number(m.group(2))
                if not np.isnan(val) and val >= 0:
                    if m.group(1) and m.group(1).upper() in {"USD", "U$S"}:
                        val = val * tipo_cambio * factor_usd
                    return val, "structured"

    desc = f"{item.get('description') or ''} {item.get('list_description') or ''}"
    if re.search(r"(?:expensas\s+incluidas|incluye\s+expensas|sin\s+expensas)", desc, re.I):
        return 0.0, "included_in_rent"

    m_desc = re.search(
        r"expensas\b[^$\n]{0,35}?(?:(usd|u\$s)\s*|(?:\$|ars)\s*)([\d]{1,3}(?:\.[\d]{3})+|\d{4,7})",
        desc,
        re.I,
    )
    if m_desc:
        val = parse_ar_number(m_desc.group(2))
        if not np.isnan(val) and val > 0:
            if m_desc.group(1):
                val = val * tipo_cambio * factor_usd
            return val, "regex_desc"

    return np.nan, "missing"


# Prueba rápida sobre un subconjunto
sample_exp = pd.DataFrame([
    {"listing_id": it.get("listingId"), "alquiler_ars": parse_price_ars(it.get("list_price_amount"), it.get("list_price_currency")),
     "expensas_ars": extract_expenses_ars(it)[0], "expensas_fuente": extract_expenses_ars(it)[1]}
    for it in raw_items
])
print("Fuentes de extracción de expensas (sobre registros crudos):")
display(sample_exp["expensas_fuente"].value_counts().to_frame("cantidad"))

## 3. Ingeniería de Atributos NLP (`apto_mascotas` y `score_amenities`)

1. **`apto_mascotas` con prioridad de negaciones:** muchas publicaciones tienen el *tag* genérico `"Permite mascotas"` pero en el cuerpo del aviso aclaran *"No se aceptan mascotas"*. Nuestra regla NLP da prioridad a la negación explícita (`nlp_negation_override`).
2. **Amenities y Scores por Perfil:** extraemos 7 amenities binarios (`gimnasio`, `pileta`, `parrilla`, `sum/coworking`, `seguridad`, `balcón/terraza`, `cochera`) combinando `featuresListCardSimplified` y NLP, y calculamos 3 puntajes en escala `0–10`:
   - `score_amenities` (balanceado)
   - `score_amenities_soltero` (prioriza gimnasio, SUM/coworking, balcón)
   - `score_amenities_familia` (prioriza seguridad, cochera, pileta, parrilla)

In [ ]:
def extract_first_int(text: str | None) -> float:
    if not text:
        return np.nan
    m = re.search(r"(\d+)", str(text))
    return float(m.group(1)) if m else np.nan


def parse_main_features(item: dict[str, Any]) -> dict[str, float]:
    mf_map: dict[str, str] = {}
    for entry in (item.get("featuresListCardSimplified") or []):
        fid = entry.get("featureId")
        if fid and entry.get("value"):
            mf_map[str(fid)] = str(entry["value"])
    for entry in (item.get("mainFeatures") or []):
        fid = entry.get("id")
        if fid and entry.get("value"):
            mf_map[str(fid)] = str(entry["value"])

    cft5_raw = mf_map.get("CFT5", "")
    if "estrenar" in cft5_raw.lower():
        antiguedad, es_a_estrenar = 0.0, 1
    else:
        antiguedad = extract_first_int(cft5_raw)
        es_a_estrenar = 1 if antiguedad == 0 else 0

    return {
        "m2_cubiertos": extract_first_int(mf_map.get("CFT101")),
        "m2_totales": extract_first_int(mf_map.get("CFT100")),
        "ambientes": extract_first_int(mf_map.get("CFT1")),
        "dormitorios": extract_first_int(mf_map.get("CFT2")),
        "banos": extract_first_int(mf_map.get("CFT3")),
        "toilettes": extract_first_int(mf_map.get("CFT4")),
        "cocheras": extract_first_int(mf_map.get("CFT7")) if "CFT7" in mf_map else 0.0,
        "antiguedad_anios": antiguedad,
        "es_a_estrenar": float(es_a_estrenar),
    }


def extract_apto_mascotas(item: dict[str, Any], return_reason: bool = False) -> int | tuple[int, str]:
    desc = f"{item.get('description') or ''} {item.get('list_description') or ''} {item.get('title') or ''}"
    pills = (item.get("detailExtraTopLevel") or {}).get("pills") or []
    features = item.get("featuresListCardSimplified") or []
    struct_yes = any(p.get("id") == "ADMITS_PETS" for p in pills) or any(
        str(fl.get("featureId")) == "1000002" or "permite mascotas" in str(fl.get("featureLabel", "")).lower()
        for fl in features
    )

    if re.search(
        r"\bno\s+(?:se\s+)?(?:acepta\w*|permit\w*|admit\w*)\s+mascotas?\b|\bprohibid\w*\s+mascotas?\b|\bsin\s+mascotas?\b",
        desc,
        re.I,
    ):
        reason = "nlp_negation_override" if struct_yes else "nlp_negation"
        return (0, reason) if return_reason else 0

    if struct_yes:
        return (1, "structured") if return_reason else 1

    if re.search(r"\b(?:apto|acepta\w*|permit\w*|admit\w*)\s+mascotas?\b|\bpet\s*friendly\b", desc, re.I):
        return (1, "nlp_positive") if return_reason else 1

    return (0, "not_specified") if return_reason else 0


def extract_amenities(item: dict[str, Any]) -> dict[str, float]:
    features = item.get("featuresListCardSimplified") or []
    f_ids = {str(fl.get("featureId")) for fl in features if fl.get("featureId")}
    f_labels = " ".join(str(fl.get("featureLabel") or "") for fl in features).lower()
    desc = f"{item.get('description') or ''} {item.get('list_description') or ''}".lower()
    text_all = f"{f_labels} {desc}"
    mf = parse_main_features(item)

    has_gimnasio = int("1000068" in f_ids or bool(re.search(r"\bgimnasio\b|\bgym\b|\bfitness\b", text_all)))
    has_pileta = int("1000079" in f_ids or bool(re.search(r"\bpileta\b|\bpiscina\b|\bswimming\b", text_all)))
    has_parrilla = int("1000078" in f_ids or "1000085" in f_ids or bool(re.search(r"\bparrilla\b|\bquincho\b", text_all)))
    has_sum = int("1000092" in f_ids or bool(re.search(r"\bsum\b|\bcoworking\b|\bco-working\b|\bsal[oó]n de usos\b", text_all)))
    has_seguridad = int(
        "1000097" in f_ids or "2000142" in f_ids or "1000039" in f_ids
        or bool(re.search(r"\bseguridad\b|\bvigilancia\b|\btótem\b|\btotem\b|\b24\s*hs\b|\b24\s*horas\b|\bencargado\b", text_all))
    )
    has_balcon_terraza = int(
        "1000100" in f_ids or "1000116" in f_ids or "1000114" in f_ids
        or bool(re.search(r"\bbalc[oó]n\b|\bterraza\b|\bpatio\b", text_all))
    )
    has_cochera = int(
        (mf["cocheras"] > 0) or "1000018" in f_ids
        or (bool(re.search(r"\bcochera\b|\bgarage\b", text_all)) and not bool(re.search(r"\bsin\s+cochera\b", text_all)))
    )

    score_soltero = 2.5 * has_gimnasio + 2.0 * has_sum + 1.5 * has_balcon_terraza + 1.5 * has_seguridad + 1.0 * has_pileta + 0.75 * has_parrilla + 0.75 * has_cochera
    score_familia = 2.5 * has_seguridad + 2.0 * has_cochera + 1.5 * has_pileta + 1.5 * has_parrilla + 1.25 * has_balcon_terraza + 0.75 * has_sum + 0.5 * has_gimnasio
    score_general = round(1.5 * has_seguridad + 1.5 * has_cochera + 1.5 * has_pileta + 1.5 * has_gimnasio + 1.5 * has_parrilla + 1.5 * has_sum + 1.0 * has_balcon_terraza, 2)

    return {
        "has_gimnasio": has_gimnasio, "has_pileta": has_pileta, "has_parrilla": has_parrilla,
        "has_sum_coworking": has_sum, "has_seguridad": has_seguridad, "has_balcon_terraza": has_balcon_terraza,
        "has_cochera": has_cochera,
        "score_amenities_soltero": round(score_soltero, 2),
        "score_amenities_familia": round(score_familia, 2),
        "score_amenities": score_general,
    }


def extract_barrio_subzona(item: dict[str, Any]) -> tuple[str, str]:
    loc = item.get("location") or {}
    raw = str(loc.get("neighborhood") or item.get("list_location_label") or "").strip()
    parts = [p.strip() for p in raw.split(",") if p.strip()]
    if len(parts) >= 2 and parts[-1].lower() in {"capital federal", "caba", "buenos aires"}:
        barrio = parts[-2]
        subzona = parts[0] if len(parts) >= 3 else barrio
    elif parts:
        barrio, subzona = parts[-1], parts[0]
    else:
        barrio, subzona = "Desconocido", "Desconocido"
    return barrio, subzona

print("Funciones NLP de mascotas, amenities y ubicación definidas correctamente.")

## 4. Construcción de Tabla Base, Limpieza Espacial e Imputación de Nulos

En este paso aplicamos las reglas de curación sobre Zonaprop:
1. Rescate de coordenadas aproximadas desde `location` (`ubicacion_aproximada = 1`).
2. Deduplicación por `listing_id` y descarte de precios inválidos (`<= 0` o `NaN`).
3. Filtrado estricto al **bounding box de CABA** para evitar que puntos en otras provincias corrompan el cruce espacial.
4. Reparación de superficies (`m2_cubiertos <= m2_totales`), imputación física (`dormitorios`, `banos`, `antiguedad_anios`) e imputación de `expensas_ars` por mediana de `expensas/m²` del mismo barrio (`expensas_imputadas = 1`).

In [ ]:
records = []
for it in raw_items:
    loc = it.get("location") or {}
    lat_raw, lon_raw = it.get("list_latitude"), it.get("list_longitude")
    ubic_aprox = 0
    if lat_raw is None or lon_raw is None:
        lat_raw, lon_raw = loc.get("latitude"), loc.get("longitude")
        ubic_aprox = 1

    alquiler_ars = parse_price_ars(it.get("list_price_amount"), it.get("list_price_currency"))
    exp_val, exp_src = extract_expenses_ars(it)
    pet_val, pet_reason = extract_apto_mascotas(it, return_reason=True)
    mf = parse_main_features(it)
    am = extract_amenities(it)
    barrio, subzona = extract_barrio_subzona(it)

    records.append({
        "listing_id": str(it.get("listingId")),
        "latitud": float(lat_raw) if lat_raw is not None else np.nan,
        "longitud": float(lon_raw) if lon_raw is not None else np.nan,
        "ubicacion_aproximada": ubic_aprox,
        "barrio": barrio,
        "subzona": subzona,
        "moneda_origen": str(it.get("list_price_currency") or "ARS"),
        "precio_origen": it.get("list_price_amount"),
        "alquiler_ars": alquiler_ars,
        "expensas_ars": exp_val,
        "expensas_fuente": exp_src,
        "apto_mascotas": pet_val,
        "mascotas_fuente": pet_reason,
        **mf,
        **am,
    })

df = pd.DataFrame(records)
audit_zp = {"raw_records": int(len(df))}

# 1. Deduplicar
dups = int(df.duplicated(subset=["listing_id"]).sum())
df = df.drop_duplicates(subset=["listing_id"], keep="first").reset_index(drop=True)
audit_zp["duplicates_removed"] = dups

# 2. Filtrar precios inválidos
invalid_price = df["alquiler_ars"].isna() | (df["alquiler_ars"] <= 0)
audit_zp["dropped_invalid_price"] = int(invalid_price.sum())
df = df.loc[~invalid_price].copy()

# 3. Filtrar coordenadas fuera de CABA
in_caba = df["latitud"].between(CABA_LAT_MIN, CABA_LAT_MAX) & df["longitud"].between(CABA_LON_MIN, CABA_LON_MAX)
audit_zp["dropped_out_of_caba_coords"] = int((~in_caba).sum())
audit_zp["out_of_caba_listing_ids"] = df.loc[~in_caba, "listing_id"].tolist()
df = df.loc[in_caba].copy().reset_index(drop=True)

# 4. Reparar e imputar dimensiones físicas
nulos_antes = df[["m2_totales", "m2_cubiertos", "ambientes", "dormitorios", "banos", "antiguedad_anios", "expensas_ars"]].isna().sum().to_dict()
audit_zp["missing_before_imputation"] = {k: int(v) for k, v in nulos_antes.items()}

ratio_cub_tot = (df["m2_cubiertos"] / df["m2_totales"]).replace([np.inf, -np.inf], np.nan).median()
if pd.isna(ratio_cub_tot) or not (0.5 <= ratio_cub_tot <= 1.0):
    ratio_cub_tot = 0.90
df["m2_totales"] = df["m2_totales"].fillna(df["m2_cubiertos"])
df["m2_cubiertos"] = df["m2_cubiertos"].fillna((df["m2_totales"] * ratio_cub_tot).round(0))

invalid_m2 = df["m2_totales"].isna() | (df["m2_totales"] <= 10) | (df["m2_totales"] > 2000)
audit_zp["dropped_invalid_m2"] = int(invalid_m2.sum())
df = df.loc[~invalid_m2].copy().reset_index(drop=True)

incoherent_m2 = df["m2_cubiertos"] > df["m2_totales"]
audit_zp["fixed_covered_gt_total_m2"] = int(incoherent_m2.sum())
df.loc[incoherent_m2, "m2_cubiertos"] = df.loc[incoherent_m2, "m2_totales"]

df["ambientes"] = df["ambientes"].fillna(df["ambientes"].median()).clip(lower=1)
missing_dorm = df["dormitorios"].isna()
df.loc[missing_dorm, "dormitorios"] = (df.loc[missing_dorm, "ambientes"] - 1).clip(lower=0)
df["banos"] = df["banos"].fillna(df.groupby("ambientes")["banos"].transform("median")).fillna(1.0)
df["antiguedad_anios"] = df["antiguedad_anios"].fillna(df.groupby("barrio")["antiguedad_anios"].transform("median")).fillna(df["antiguedad_anios"].median())
df["toilettes"] = df["toilettes"].fillna(0.0)

# 5. Imputación de expensas por mediana de expensas/m2 del barrio
df["expensas_imputadas"] = df["expensas_ars"].isna().astype(int)
valid_exp = df["expensas_ars"].notna() & (df["expensas_ars"] > 0)
exp_m2 = pd.Series(np.nan, index=df.index)
exp_m2.loc[valid_exp] = df.loc[valid_exp, "expensas_ars"] / df.loc[valid_exp, "m2_totales"]
med_exp_m2_barrio = exp_m2.groupby(df["barrio"]).transform("median").fillna(float(exp_m2.median()) if exp_m2.notna().any() else 2500.0)
df.loc[df["expensas_ars"].isna(), "expensas_ars"] = (df.loc[df["expensas_ars"].isna(), "m2_totales"] * med_exp_m2_barrio.loc[df["expensas_ars"].isna()]).round(0)
df["expensas_ars"] = df["expensas_ars"].clip(lower=0.0)

# 6. Target y métricas por m2
df["costo_total_ars"] = df["alquiler_ars"] + df["expensas_ars"]
df["precio_m2_total_ars"] = df["costo_total_ars"] / df["m2_totales"]
df["precio_m2_cubierto_ars"] = df["costo_total_ars"] / df["m2_cubiertos"].clip(lower=1.0)

audit_zp["expensas_source_counts"] = df["expensas_fuente"].value_counts().to_dict()
audit_zp["mascotas_source_counts"] = df["mascotas_fuente"].value_counts().to_dict()
audit_zp["valid_properties_after_etl"] = int(len(df))

print("Resumen Auditoría Zonaprop:")
print(f"  - Registros crudos         : {audit_zp['raw_records']}")
print(f"  - Duplicados removidos     : {audit_zp['duplicates_removed']}")
print(f"  - Descartados precio <= 0  : {audit_zp['dropped_invalid_price']}")
print(f"  - Fuera de CABA descartados: {audit_zp['dropped_out_of_caba_coords']} -> {audit_zp['out_of_caba_listing_ids']}")
print(f"  - m2 inválidos descartados : {audit_zp['dropped_invalid_m2']}")
print(f"  - Propiedades válidas      : {audit_zp['valid_properties_after_etl']}")
print("\nAuditoría de etiquetas de mascotas:")
display(df["mascotas_fuente"].value_counts().to_frame("cantidad"))

## 5. Enriquecimiento Geoespacial con las 4 Capas Urbanas de CABA

Cruzamos las **964 propiedades válidas** con cuatro fuentes oficiales de la Ciudad de Buenos Aires usando el sistema de coordenadas unificado **WGS84 (`EPSG:4326`)** y distancias geodésicas (**Haversine** vía `BallTree`).

### 5.1 Capa Acústica: Mapa de Ruido Diurno (`medicion_de_ruido_diurno_2018.csv`)
Los polígonos de ruido modelan principalmente **calzadas y avenidas**; por ello combinamos `gpd.sjoin(..., predicate="within")` con respaldo `gpd.sjoin_nearest` en proyección métrica local de CABA (`ruido_imputado_nearest`).

In [ ]:
try:
    csv.field_size_limit(sys.maxsize)
except OverflowError:
    csv.field_size_limit(2_147_483_647)

ruido_df = pd.read_csv(RUIDO_PATH)
ruido_df["geometry"] = ruido_df["wkt"].apply(wkt.loads)
ruido_df["ruido_db_promedio"] = (ruido_df["limite_inferior_rango_db"].astype(float) + ruido_df["limite_superior_rango_db"].astype(float)) / 2.0
gdf_ruido = gpd.GeoDataFrame(ruido_df, geometry="geometry", crs="EPSG:4326")[["ruido_db_promedio", "comuna", "rango_db", "geometry"]]

gdf_props = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df["longitud"], df["latitud"]), crs="EPSG:4326")
joined = gpd.sjoin(gdf_props, gdf_ruido, how="left", predicate="within")
if joined.index.duplicated().any():
    joined = joined.groupby(joined.index).agg({**{c: "first" for c in df.columns}, "geometry": "first", "ruido_db_promedio": "mean", "comuna": "first", "rango_db": "first"})
    joined = gpd.GeoDataFrame(joined, geometry="geometry", crs="EPSG:4326")

missing_mask = joined["ruido_db_promedio"].isna()
n_nearest = int(missing_mask.sum())
if n_nearest > 0:
    sub_missing = gdf_props.loc[missing_mask, ["listing_id", "geometry"]].to_crs(CABA_LOCAL_PROJ)
    ruido_proj = gdf_ruido.to_crs(CABA_LOCAL_PROJ)
    nearest = gpd.sjoin_nearest(sub_missing, ruido_proj, how="left", distance_col="dist_poligono_ruido_m")
    nearest = nearest.groupby(nearest.index).first()
    joined.loc[missing_mask, "ruido_db_promedio"] = nearest["ruido_db_promedio"].values
    joined.loc[missing_mask, "comuna"] = nearest["comuna"].values
    joined.loc[missing_mask, "rango_db"] = nearest["rango_db"].values

df = pd.DataFrame(joined.drop(columns=["geometry", "index_right"], errors="ignore"))
df["comuna"] = pd.to_numeric(df["comuna"], errors="coerce").fillna(df["barrio"].map(BARRIO_COMUNA_FALLBACK)).fillna(0).astype(int)
df["ruido_imputado_nearest"] = missing_mask.astype(int).values

audit_ruido = {"ruido_polygons": int(len(gdf_ruido)), "matched_within": int(len(df) - n_nearest), "imputed_nearest": n_nearest}
print("Auditoría Capa Ruido:", audit_ruido)

### 5.2 Capa de Seguridad: Delitos 2022 (`delitos_2022.xlsx`)
El dataset de delitos contiene coordenadas con **punto decimal desplazado** (ej. `-34584136.0`). Aplicamos `normalize_caba_coord()` (división iterativa por 10 hasta entrar en el rango CABA), calculamos `peso_gravedad` (Homicidio `10`, Arma `4`, Robo/Lesiones `2`, Otros `1`) y consultamos con `BallTree(metric="haversine")` en un radio de **500 metros**.

In [ ]:
def normalize_caba_coord(val: Any, min_val: float, max_val: float) -> float:
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    try:
        if isinstance(val, str):
            val = val.strip().replace(",", ".")
            if not val:
                return np.nan
        num = float(val)
    except (TypeError, ValueError):
        return np.nan
    if not np.isfinite(num) or num == 0.0:
        return np.nan
    if min_val <= num <= max_val:
        return num
    if num > 0:
        return np.nan
    temp = num
    for _ in range(25):
        if min_val <= temp <= max_val:
            return temp
        if abs(temp) < abs(max_val):
            break
        temp /= 10.0
    temp = num
    for _ in range(5):
        if min_val <= temp <= max_val:
            return temp
        if abs(temp) > abs(min_val):
            break
        temp *= 10.0
    return np.nan


delitos = pd.read_excel(DELITOS_PATH)
raw_del_count = int(len(delitos))
null_coords = int((delitos["latitud"].isna() | delitos["longitud"].isna()).sum())

delitos["lat_norm"] = delitos["latitud"].apply(lambda v: normalize_caba_coord(v, CABA_LAT_MIN, CABA_LAT_MAX))
delitos["lon_norm"] = delitos["longitud"].apply(lambda v: normalize_caba_coord(v, CABA_LON_MIN, CABA_LON_MAX))
valid_del = delitos.dropna(subset=["lat_norm", "lon_norm"]).copy()
unrecoverable = int(raw_del_count - len(valid_del) - null_coords)

tipo_str = valid_del["tipo"].astype(str).str.strip().str.lower()
arma_si = valid_del["uso_arma"].astype(str).str.strip().str.upper() == "SI"
cant = pd.to_numeric(valid_del["cantidad"], errors="coerce").fillna(1.0).clip(lower=1.0)
base_weight = np.where(
    tipo_str == "homicidios", CRIME_WEIGHT_HOMICIDIO,
    np.where(arma_si, CRIME_WEIGHT_ARMA, np.where(tipo_str.isin(["robo", "lesiones"]), CRIME_WEIGHT_ROBO_LESIONES, CRIME_WEIGHT_OTROS))
)
valid_del["peso_gravedad"] = base_weight * cant
valid_del["es_robo_hurto"] = tipo_str.isin(["robo", "hurto"]).astype(float) * cant
valid_del["cantidad_clean"] = cant

tree_del = BallTree(np.radians(valid_del[["lat_norm", "lon_norm"]].to_numpy()), metric="haversine")
prop_coords_rad = np.radians(df[["latitud", "longitud"]].to_numpy())
neighbors_del = tree_del.query_radius(prop_coords_rad, r=500.0 / EARTH_RADIUS_M)

cant_arr = valid_del["cantidad_clean"].to_numpy()
robo_arr = valid_del["es_robo_hurto"].to_numpy()
grav_arr = valid_del["peso_gravedad"].to_numpy()

df["densidad_delitos_500m"] = [float(cant_arr[idx].sum()) for idx in neighbors_del]
df["densidad_robos_500m"] = [float(robo_arr[idx].sum()) for idx in neighbors_del]
df["indice_gravedad_delitos_500m"] = [float(grav_arr[idx].sum()) for idx in neighbors_del]

audit_delitos = {
    "delitos_raw_rows": raw_del_count, "delitos_null_coords": null_coords,
    "delitos_unrecoverable_coords": unrecoverable, "delitos_valid_rows": int(len(valid_del)),
}
print("Auditoría Capa Delitos:", audit_delitos)

### 5.3 Capa Educativa (`establecimientos_educativos.xlsx`) y 5.4 Red de Subtes (`bocas-de-subte.csv`)

- **Escuelas:** vienen en proyección local Gauss-Krüger de CABA (`CABA_LOCAL_PROJ`), por lo que las reproyectamos a `EPSG:4326` y filtramos los niveles familiares (`Nivel inicial`, `Nivel primario`, `Nivel secundario común`). Calculamos cantidad a 500 m y distancia mínima.
- **Subte:** calculamos distancia mínima a cualquier boca (`distancia_subte_m`) y cantidad de líneas distintas a menos de 600 m (`lineas_subte_600m`).

In [ ]:
# 5.3 Establecimientos educativos
esc = pd.read_excel(ESCUELAS_PATH)
esc_raw_count = int(len(esc))
esc["geometry"] = esc["geometry"].apply(wkt.loads)
gdf_esc = gpd.GeoDataFrame(esc, geometry="geometry", crs=CABA_LOCAL_PROJ).to_crs("EPSG:4326")
gdf_esc["lat"] = gdf_esc.geometry.y
gdf_esc["lon"] = gdf_esc.geometry.x

pattern_esc = r"Nivel inicial|Nivel primario|Nivel secundario común"
gdf_fam = gdf_esc.loc[gdf_esc["nen_mde"].astype(str).str.contains(pattern_esc, case=False, na=False, regex=True)].dropna(subset=["lat", "lon"]).copy()

tree_esc = BallTree(np.radians(gdf_fam[["lat", "lon"]].to_numpy()), metric="haversine")
counts_esc = tree_esc.query_radius(prop_coords_rad, r=500.0 / EARTH_RADIUS_M, count_only=True)
dists_esc_rad, _ = tree_esc.query(prop_coords_rad, k=1)

df["escuelas_familias_500m"] = counts_esc.astype(int)
df["distancia_escuela_m"] = (dists_esc_rad[:, 0] * EARTH_RADIUS_M).round(2)

audit_escuelas = {"escuelas_raw_rows": esc_raw_count, "escuelas_familias_filtered": int(len(gdf_fam))}
print("Auditoría Escuelas:", audit_escuelas)

# 5.4 Bocas de Subte
subte = pd.read_csv(SUBTE_PATH).dropna(subset=["lat", "long"]).copy()
tree_subte = BallTree(np.radians(subte[["lat", "long"]].to_numpy()), metric="haversine")
dists_sub_rad, _ = tree_subte.query(prop_coords_rad, k=1)
neighbors_sub = tree_subte.query_radius(prop_coords_rad, r=600.0 / EARTH_RADIUS_M)

lineas_arr = subte["linea"].astype(str).str.strip().to_numpy()
df["distancia_subte_m"] = (dists_sub_rad[:, 0] * EARTH_RADIUS_M).round(2)
df["lineas_subte_600m"] = [int(len(set(lineas_arr[idx]))) for idx in neighbors_sub]

audit_subte = {"subte_bocas_count": int(len(subte)), "unique_lines_total": sorted(subte["linea"].astype(str).str.strip().unique().tolist())}
print("Auditoría Subte:", audit_subte)

# Mapa geográfico de verificación visual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc1 = axes[0].scatter(df["longitud"], df["latitud"], c=df["precio_m2_total_ars"] / 1000, cmap="viridis", s=16, alpha=0.8)
axes[0].set_title("Distribución Espacial: Precio/m² (Miles ARS)")
axes[0].set_xlabel("Longitud")
axes[0].set_ylabel("Latitud")
fig.colorbar(sc1, ax=axes[0], label="Miles ARS / m²")

sc2 = axes[1].scatter(df["longitud"], df["latitud"], c=df["ruido_db_promedio"], cmap="magma_r", s=16, alpha=0.8)
axes[1].set_title("Distribución Espacial: Ruido Diurno Asignado (dBA)")
axes[1].set_xlabel("Longitud")
axes[1].set_ylabel("Latitud")
fig.colorbar(sc2, ax=axes[1], label="dBA")
fig.tight_layout()
plt.show()

## 6. Auditoría de Outliers (IQR `1.5×` y `|Z-score| > 3`)

Detectamos valores extremos en `costo_total_ars`, `precio_m2_total_ars` y `m2_totales` generando **banderas de auditoría** (`outlier_iqr_*`, `outlier_zscore_*`) **sin eliminar** propiedades reales de segmento alto para no sesgar el mercado inmobiliario.

In [ ]:
outlier_cols = ["costo_total_ars", "precio_m2_total_ars", "m2_totales"]
outlier_summary = {}

for col in outlier_cols:
    s = df[col].astype(float)
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    low_iqr, high_iqr = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flag_iqr = (s < low_iqr) | (s > high_iqr)

    std = float(s.std(ddof=1))
    z = (s - float(s.mean())) / std if std > 0 else pd.Series(0.0, index=s.index)
    flag_z = z.abs() > 3.0

    df[f"outlier_iqr_{col}"] = flag_iqr.astype(int)
    df[f"outlier_zscore_{col}"] = flag_z.astype(int)
    outlier_summary[col] = {
        "Q1": round(q1, 2), "Q3": round(q3, 2), "IQR": round(iqr, 2),
        "Techo_IQR": round(high_iqr, 2),
        "Outliers_IQR": int(flag_iqr.sum()), "Pct_IQR": round(float(flag_iqr.mean() * 100.0), 2),
        "Outliers_Z3": int(flag_z.sum()), "Pct_Z3": round(float(flag_z.mean() * 100.0), 2),
    }

display(pd.DataFrame(outlier_summary).T)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes, outlier_cols, ["Costo Total (ARS)", "Precio/m² (ARS)", "m² Totales"]):
    sns.boxplot(y=df[col], ax=ax, color="#4c72b0")
    ax.set_title(label)
fig.tight_layout()
plt.show()

## 7. Exportación de Resultados Curados y Resumen Ejecutivo

Guardamos el dataset enriquecido completo (`producto1/dataset_enriquecido_completo.csv`) y mostramos la vista previa final lista para la **Fase 2 (EDA Estadístico y Selección de Features)**.

In [ ]:
output_enriched = PRODUCTO1_DIR / "dataset_enriquecido_completo.csv"
df.to_csv(output_enriched, index=False, encoding="utf-8")

print(f"Dataset enriquecido guardado en: {output_enriched}")
print(f"Dimensiones finales: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Nulos totales en el dataset final: {int(df.isna().sum().sum())}")
display(df.head(5))